# 09 · Análisis exploratorio de series temporales

**Qué pregunta responde.** ¿Qué estructura tiene el corpus antes de modelar, y qué decisiones de diseño se derivan de ella? Aplica las técnicas de la asignatura de series temporales (estacionariedad, autocorrelación, descomposición estacional, espectro).

**Qué entradas usa.** `results/eda_series_temporales.csv` (métricas agregadas sobre una muestra de 285–300 series del corpus) y las figuras en `reports/memoria/figures/fig_eda_*.png`. No requiere el corpus privado.

**Qué produce.** Cinco lecturas, cada una con su conclusión operativa: estacionariedad (ADF/KPSS), autocorrelación (ACF), estacionalidad (STL), ciclo horario (Fourier) y calidad/imputación.

**Conclusión (2 frases).** El precio medio es integrado de orden uno (modelar sobre diferencias, nunca sobre niveles) y los retornos carecen de estructura lineal apreciable, pero el estado del libro es muy persistente y la volatilidad se agrupa: la señal está en el libro, no en el retorno retardado. El ciclo horario es real (minuto 50 el más activo) y la malla de 2 s tiene cobertura casi perfecta, lo que permite una imputación conservadora.

**Fila de `results/key_results.csv`.** Ninguna propia (ese registro recoge el arco experimental original): la evidencia numérica de este cuaderno vive en `results/eda_series_temporales.csv`, y sus decisiones gobiernan el diseño de features y la imputación de todo el arco (memoria §3.2).

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image
ROOT = Path.cwd()
if not (ROOT / 'results').exists(): ROOT = ROOT.parent
eda = pd.read_csv(ROOT / 'results' / 'eda_series_temporales.csv')
eda


## Estacionariedad (ADF / KPSS)
ADF rechaza la raíz unitaria en solo ~6 % de las series en niveles pero en el 100 % en diferencias; KPSS, al revés. Ambos coinciden: **el mid es I(1)**. Decisión: modelar sobre diferencias/retornos.

![Estacionariedad](../reports/memoria/figures/fig_eda_estacionariedad.png)


## Autocorrelación (ACF)
ACF(1) de retornos ≈ 0,07 (sin apenas estructura lineal), pero |retornos| y el desequilibrio de libro son muy persistentes (0,15 y 0,99). Decisión: buscar la señal en el estado del libro y la volatilidad, no en el retorno retardado.

![ACF](../reports/memoria/figures/fig_eda_acf.png)


## Estacionalidad (STL) y ciclo horario (Fourier)
La componente estacional explica ~19 % de la varianza de la actividad; el espectro tiene picos nítidos en 24 h y 1 h, y el **minuto 50** es el más activo del ciclo horario — coherente con la concentración de actividad en el tramo terminal.

![STL](../reports/memoria/figures/fig_eda_stl.png)

![Fourier](../reports/memoria/figures/fig_eda_fourier.png)


## Calidad e imputación
Cobertura mediana del 100 % y >99 % en casi todas las series: la densidad permite arrastrar el último valor marcándolo `stale`, sin interpolar (interpolar inventaría microestructura).

![Calidad](../reports/memoria/figures/fig_eda_calidad.png)
